# Imbalanced Learning version 2

Let's try doing hyper-parameter optimization based on balanced acuracy (i.e. macro recall) for each method.

## Data Preparation

In [1]:
%matplotlib inline

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from collections import Counter
from collections import defaultdict
from sklearn.preprocessing import StandardScaler

In [2]:
#df = pd.read_csv('cmi_internet_cleaned.csv')
df = pd.read_csv('../dataset/cmi_internet_cleaned.csv')
#df

In [3]:
X = df.drop(columns=['sii'])
y = df['sii'].astype(int)

num_cols = X.select_dtypes(include=np.number).columns.tolist()
print("Variables used:")
print(num_cols)
print("Shape:", X.shape)

Variables used:
['Basic_Demos-Age', 'Basic_Demos-Sex', 'CGAS-CGAS_Score', 'Physical-Height', 'Physical-Weight', 'Physical-Waist_Circumference', 'Physical-Diastolic_BP', 'Fitness_Endurance-Max_Stage', 'Physical-HeartRate', 'Physical-Systolic_BP', 'FGC-FGC_CU', 'FGC-FGC_GSND', 'FGC-FGC_GSD', 'FGC-FGC_PU', 'FGC-FGC_SRL', 'FGC-FGC_SRR', 'FGC-FGC_TL', 'BIA-BIA_DEE', 'BIA-BIA_Activity_Level_num', 'BIA-BIA_BMC', 'BIA-BIA_BMR', 'BIA-BIA_ECW', 'BIA-BIA_Fat', 'BIA-BIA_Frame_num', 'BIA-BIA_ICW', 'BIA-BIA_LDM', 'BIA-BIA_LST', 'BIA-BIA_SMM', 'PAQ_Total', 'SDS-SDS_Total_T', 'PreInt_EduHx-computerinternet_hoursday', 'Fitness_Endurance-Time']
Shape: (7839, 32)


In [4]:
scaler = StandardScaler() # very important for KNN

X = scaler.fit_transform(X)

X = pd.DataFrame(
    X,
    columns=num_cols,
    index=df.index
)

#X

In [5]:
# The four classes of sii are imbalanced:
ctr = Counter(y)
ctr

Counter({0: 5476, 1: 1444, 2: 846, 3: 73})

In [6]:
5476/(5476+1444+846+73), 73/(5476+1444+846+73)

(0.6985584896032657, 0.009312412297486925)

## Data Partitioning

In [7]:
from sklearn.model_selection import train_test_split, cross_val_score 

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=100, stratify=y)

In [9]:
# Records of each class in train and test datasets keep the proportion of the original dataset thanks to 'stratify':
np.unique(y_train, return_counts=True), np.unique(y_test, return_counts=True)

((array([0, 1, 2, 3]), array([3833, 1011,  592,   51])),
 (array([0, 1, 2, 3]), array([1643,  433,  254,   22])))

## Basic Classification

In [10]:
from sklearn.metrics import accuracy_score, f1_score, classification_report
#from sklearn.metrics import roc_curve, auc, roc_auc_score

In [11]:
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

In [12]:
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import RepeatedStratifiedKFold

In [13]:
import warnings
warnings.simplefilter("ignore")

In [14]:
clf = DummyClassifier() # classifies everything as the majority class (class 0 in this case)
clf.fit(X_train, y_train)

y_pred0 = clf.predict(X_test)
print(classification_report(y_test, y_pred0))

              precision    recall  f1-score   support

           0       0.70      1.00      0.82      1643
           1       0.00      0.00      0.00       433
           2       0.00      0.00      0.00       254
           3       0.00      0.00      0.00        22

    accuracy                           0.70      2352
   macro avg       0.17      0.25      0.21      2352
weighted avg       0.49      0.70      0.57      2352



In [15]:
# Hyper-parameter tuning for Decision Tree Classifier
# Default: gini criterion, no max depth, min_samples_split=2, min_samples_leaf=1, no class weights

param_list = {
    'max_depth': [None] + list(np.arange(2, 20)),
    'min_samples_split': [2, 5, 10, 20, 30, 50],
    'min_samples_leaf': [1, 5, 10, 20, 30, 50],
    'criterion': ['gini', 'entropy']
}

def doDT(X,y):
    random_search = RandomizedSearchCV(
        DecisionTreeClassifier(),
        param_distributions=param_list,
        scoring='balanced_accuracy',
        cv=RepeatedStratifiedKFold(random_state=0),
        n_jobs=-1,
        refit=True,
        n_iter=200,
        #verbose=2
    )

    random_search.fit(X, y)
    clf = random_search.best_estimator_

    print(random_search.best_params_, random_search.best_score_)
    clf.fit(X, y)

    y_pred0 = clf.predict(X_test)
    print(classification_report(y_test, y_pred0))

In [16]:
doDT(X_train,y_train)

{'min_samples_split': 30, 'min_samples_leaf': 1, 'max_depth': None, 'criterion': 'gini'} 0.29932497989989065
              precision    recall  f1-score   support

           0       0.74      0.81      0.78      1643
           1       0.27      0.22      0.24       433
           2       0.26      0.19      0.22       254
           3       0.17      0.05      0.07        22

    accuracy                           0.63      2352
   macro avg       0.36      0.32      0.33      2352
weighted avg       0.60      0.63      0.61      2352



In [17]:
# Hyper-parameter tuning for KNN Classifier
# Default: 5 neighbours, minkowski distance, uniform weights

param_grid = {
    "n_neighbors": [e for e in range(1, 50) if e % 2 != 0],
    "weights": ["uniform", "distance"],
    "metric": ["minkowski", "cityblock"],
}

def doKNN(X,y):
    random_search = RandomizedSearchCV(
        KNeighborsClassifier(),
        param_distributions=param_grid,
        scoring='balanced_accuracy',
        cv=RepeatedStratifiedKFold(random_state=0),
        n_jobs=-1,
        refit=True,
        n_iter=200,
        #verbose=2
    )

    random_search.fit(X, y)
    clf = random_search.best_estimator_

    print(random_search.best_params_, random_search.best_score_)
    clf.fit(X, y)

    y_pred0 = clf.predict(X_test)
    print(classification_report(y_test, y_pred0))

In [18]:
doKNN(X_train,y_train)

{'weights': 'uniform', 'n_neighbors': 1, 'metric': 'cityblock'} 0.2937375478705999
              precision    recall  f1-score   support

           0       0.73      0.78      0.76      1643
           1       0.22      0.20      0.21       433
           2       0.18      0.13      0.15       254
           3       0.04      0.05      0.04        22

    accuracy                           0.60      2352
   macro avg       0.29      0.29      0.29      2352
weighted avg       0.57      0.60      0.58      2352



## Undersampling

In [20]:
from imblearn.under_sampling import RandomUnderSampler
from imblearn.under_sampling import CondensedNearestNeighbour
from imblearn.under_sampling import TomekLinks
from imblearn.under_sampling import EditedNearestNeighbours

### RandomUnderSampler

In [21]:
rus = RandomUnderSampler(random_state=42)
X_res, y_res = rus.fit_resample(X_train, y_train)
print('Resampled dataset shape %s' % Counter(y_res))

Resampled dataset shape Counter({0: 51, 1: 51, 2: 51, 3: 51})


In [22]:
doDT(X_res,y_res)

{'min_samples_split': 50, 'min_samples_leaf': 1, 'max_depth': 4, 'criterion': 'gini'} 0.31018181818181817
              precision    recall  f1-score   support

           0       0.77      0.33      0.46      1643
           1       0.20      0.12      0.15       433
           2       0.11      0.27      0.16       254
           3       0.02      0.64      0.03        22

    accuracy                           0.29      2352
   macro avg       0.28      0.34      0.20      2352
weighted avg       0.59      0.29      0.37      2352



In [23]:
doKNN(X_res,y_res)

{'weights': 'uniform', 'n_neighbors': 19, 'metric': 'cityblock'} 0.3321818181818182
              precision    recall  f1-score   support

           0       0.75      0.38      0.51      1643
           1       0.24      0.05      0.09       433
           2       0.16      0.07      0.10       254
           3       0.01      0.59      0.02        22

    accuracy                           0.29      2352
   macro avg       0.29      0.28      0.18      2352
weighted avg       0.59      0.29      0.38      2352



### CondensedNearestNeighbour

In [24]:
cnn = CondensedNearestNeighbour(random_state=42, n_jobs=10)
X_res, y_res = cnn.fit_resample(X_train, y_train)
print('Resampled dataset shape %s' % Counter(y_res))

Resampled dataset shape Counter({0: 399, 1: 278, 2: 228, 3: 51})


In [25]:
doDT(X_res,y_res)

{'min_samples_split': 50, 'min_samples_leaf': 20, 'max_depth': 12, 'criterion': 'gini'} 0.294358019816055
              precision    recall  f1-score   support

           0       0.77      0.67      0.72      1643
           1       0.25      0.28      0.26       433
           2       0.18      0.31      0.23       254
           3       0.00      0.00      0.00        22

    accuracy                           0.55      2352
   macro avg       0.30      0.32      0.30      2352
weighted avg       0.60      0.55      0.57      2352



In [26]:
doKNN(X_res,y_res)

{'weights': 'uniform', 'n_neighbors': 33, 'metric': 'minkowski'} 0.25584479132666393
              precision    recall  f1-score   support

           0       0.71      0.96      0.82      1643
           1       0.20      0.04      0.07       433
           2       0.22      0.04      0.07       254
           3       0.00      0.00      0.00        22

    accuracy                           0.68      2352
   macro avg       0.28      0.26      0.24      2352
weighted avg       0.56      0.68      0.59      2352



### Tomek Links

In [27]:
tl = TomekLinks()
X_res, y_res = tl.fit_resample(X_train, y_train)
print('Resampled dataset shape %s' % Counter(y_res))

Resampled dataset shape Counter({0: 3626, 1: 831, 2: 495, 3: 51})


In [28]:
doDT(X_res,y_res)

{'min_samples_split': 5, 'min_samples_leaf': 1, 'max_depth': 18, 'criterion': 'gini'} 0.2987674113005635
              precision    recall  f1-score   support

           0       0.73      0.80      0.76      1643
           1       0.29      0.23      0.26       433
           2       0.22      0.19      0.20       254
           3       0.00      0.00      0.00        22

    accuracy                           0.62      2352
   macro avg       0.31      0.30      0.31      2352
weighted avg       0.59      0.62      0.60      2352



In [29]:
doKNN(X_res,y_res)

{'weights': 'uniform', 'n_neighbors': 1, 'metric': 'cityblock'} 0.3095163436155863
              precision    recall  f1-score   support

           0       0.73      0.81      0.77      1643
           1       0.23      0.19      0.21       433
           2       0.18      0.11      0.14       254
           3       0.04      0.05      0.04        22

    accuracy                           0.61      2352
   macro avg       0.30      0.29      0.29      2352
weighted avg       0.57      0.61      0.59      2352



### Edited Nearest Neighbors

In [30]:
enn = EditedNearestNeighbours()
X_res, y_res = enn.fit_resample(X_train, y_train)
print('Resampled dataset shape %s' % Counter(y_res))

Resampled dataset shape Counter({0: 1967, 3: 51, 1: 7, 2: 4})


In [31]:
doDT(X_res,y_res)

{'min_samples_split': 5, 'min_samples_leaf': 1, 'max_depth': 18, 'criterion': 'entropy'} 0.30240540523614884
              precision    recall  f1-score   support

           0       0.72      0.95      0.82      1643
           1       0.33      0.01      0.01       433
           2       0.25      0.02      0.04       254
           3       0.04      0.27      0.07        22

    accuracy                           0.67      2352
   macro avg       0.34      0.31      0.23      2352
weighted avg       0.59      0.67      0.58      2352



In [32]:
doKNN(X_res,y_res)

{'weights': 'uniform', 'n_neighbors': 1, 'metric': 'minkowski'} 0.3136034125499581
              precision    recall  f1-score   support

           0       0.71      0.96      0.81      1643
           1       0.29      0.01      0.02       433
           2       0.23      0.01      0.02       254
           3       0.03      0.14      0.05        22

    accuracy                           0.67      2352
   macro avg       0.31      0.28      0.23      2352
weighted avg       0.57      0.67      0.57      2352



### Cluster Centroids

In [33]:
from sklearn.cluster import MiniBatchKMeans
from imblearn.under_sampling import ClusterCentroids

In [34]:
cc = ClusterCentroids(estimator=MiniBatchKMeans(n_init=1, random_state=0), random_state=42)

X_res, y_res = cc.fit_resample(X_train, y_train)
print('Resampled dataset shape %s' % Counter(y_res))

Resampled dataset shape Counter({0: 51, 1: 51, 2: 51, 3: 51})


In [35]:
doDT(X_res,y_res)

{'min_samples_split': 50, 'min_samples_leaf': 10, 'max_depth': 4, 'criterion': 'entropy'} 0.5716818181818182
              precision    recall  f1-score   support

           0       0.00      0.00      0.00      1643
           1       0.00      0.00      0.00       433
           2       0.17      0.20      0.18       254
           3       0.01      0.73      0.02        22

    accuracy                           0.03      2352
   macro avg       0.04      0.23      0.05      2352
weighted avg       0.02      0.03      0.02      2352



In [36]:
doKNN(X_res,y_res)

{'weights': 'distance', 'n_neighbors': 39, 'metric': 'cityblock'} 0.34459090909090906
              precision    recall  f1-score   support

           0       0.74      0.64      0.69      1643
           1       0.23      0.05      0.08       433
           2       0.21      0.24      0.22       254
           3       0.01      0.36      0.03        22

    accuracy                           0.49      2352
   macro avg       0.30      0.32      0.25      2352
weighted avg       0.58      0.49      0.52      2352



## Oversampling

In [37]:
from imblearn.over_sampling import RandomOverSampler
from imblearn.over_sampling import SMOTE
from imblearn.over_sampling import ADASYN

### RandomOverSampler

In [38]:
ros = RandomOverSampler(random_state=42)
X_res, y_res = ros.fit_resample(X_train, y_train)
print('Resampled dataset shape %s' % Counter(y_res))

Resampled dataset shape Counter({1: 3833, 0: 3833, 2: 3833, 3: 3833})


In [39]:
doDT(X_res,y_res)

{'min_samples_split': 2, 'min_samples_leaf': 1, 'max_depth': None, 'criterion': 'gini'} 0.9115445038653871
              precision    recall  f1-score   support

           0       0.73      0.71      0.72      1643
           1       0.23      0.24      0.24       433
           2       0.18      0.20      0.19       254
           3       0.04      0.05      0.04        22

    accuracy                           0.56      2352
   macro avg       0.29      0.30      0.30      2352
weighted avg       0.57      0.56      0.57      2352



In [40]:
doKNN(X_res,y_res)

{'weights': 'uniform', 'n_neighbors': 1, 'metric': 'cityblock'} 0.9319005841483384
              precision    recall  f1-score   support

           0       0.73      0.78      0.76      1643
           1       0.22      0.20      0.21       433
           2       0.18      0.13      0.15       254
           3       0.04      0.05      0.04        22

    accuracy                           0.60      2352
   macro avg       0.29      0.29      0.29      2352
weighted avg       0.57      0.60      0.58      2352



### SMOTE

In [41]:
sm = SMOTE(random_state=42)
X_res, y_res = sm.fit_resample(X_train, y_train)
print('Resampled dataset shape %s' % Counter(y_res))

Resampled dataset shape Counter({1: 3833, 0: 3833, 2: 3833, 3: 3833})


In [42]:
doDT(X_res,y_res)

{'min_samples_split': 2, 'min_samples_leaf': 1, 'max_depth': 15, 'criterion': 'entropy'} 0.7247961352936572
              precision    recall  f1-score   support

           0       0.76      0.68      0.72      1643
           1       0.25      0.29      0.26       433
           2       0.20      0.27      0.23       254
           3       0.05      0.09      0.07        22

    accuracy                           0.55      2352
   macro avg       0.31      0.33      0.32      2352
weighted avg       0.60      0.55      0.57      2352



In [43]:
doKNN(X_res,y_res)

{'weights': 'uniform', 'n_neighbors': 1, 'metric': 'cityblock'} 0.9102530628640289
              precision    recall  f1-score   support

           0       0.76      0.66      0.71      1643
           1       0.22      0.28      0.25       433
           2       0.15      0.19      0.17       254
           3       0.01      0.05      0.02        22

    accuracy                           0.53      2352
   macro avg       0.29      0.29      0.29      2352
weighted avg       0.59      0.53      0.56      2352



### ADASYN

In [44]:
ada = ADASYN(random_state=42)
X_res, y_res = ada.fit_resample(X_train, y_train)
print('Resampled dataset shape %s' % Counter(y_res))

Resampled dataset shape Counter({0: 3833, 3: 3831, 2: 3728, 1: 3715})


In [45]:
doDT(X_res,y_res)

{'min_samples_split': 2, 'min_samples_leaf': 1, 'max_depth': 18, 'criterion': 'gini'} 0.7330153804877192
              precision    recall  f1-score   support

           0       0.75      0.66      0.70      1643
           1       0.23      0.28      0.25       433
           2       0.17      0.23      0.19       254
           3       0.00      0.00      0.00        22

    accuracy                           0.54      2352
   macro avg       0.29      0.29      0.29      2352
weighted avg       0.58      0.54      0.56      2352



In [46]:
doKNN(X_res,y_res)

{'weights': 'uniform', 'n_neighbors': 1, 'metric': 'cityblock'} 0.91237647011557
              precision    recall  f1-score   support

           0       0.75      0.66      0.70      1643
           1       0.22      0.27      0.25       433
           2       0.17      0.20      0.19       254
           3       0.01      0.05      0.02        22

    accuracy                           0.53      2352
   macro avg       0.29      0.30      0.29      2352
weighted avg       0.59      0.53      0.56      2352



## Extra: Under + Over

In [ ]:
# let's try to do first oversampling and then undersampling: choosing out of the two best results for each (and also models 
# that don't end up with the same number of records (e.g. both random make no sense)):
# ADASYN and then ENN:
X_res, y_res = ada.fit_resample(X_train, y_train)
X_res, y_res = enn.fit_resample(X_res, y_res)
print('Resampled dataset shape %s' % Counter(y_res))


Resampled dataset shape Counter({3: 3830, 1: 3715, 2: 3679, 0: 1000})


In [51]:
doDT(X_res,y_res)

{'min_samples_split': 2, 'min_samples_leaf': 1, 'max_depth': None, 'criterion': 'entropy', 'class_weight': {0: 1, 1: 2, 2: 3, 3: 4}} 0.7477151222886634
              precision    recall  f1-score   support

           0       0.80      0.30      0.44      1643
           1       0.21      0.51      0.30       433
           2       0.12      0.30      0.17       254
           3       0.05      0.18      0.08        22

    accuracy                           0.34      2352
   macro avg       0.30      0.32      0.25      2352
weighted avg       0.61      0.34      0.38      2352



In [52]:
doKNN(X_res,y_res)

{'weights': 'uniform', 'n_neighbors': 1, 'metric': 'minkowski'} 0.9552787519485819
              precision    recall  f1-score   support

           0       0.84      0.31      0.45      1643
           1       0.21      0.47      0.29       433
           2       0.14      0.36      0.20       254
           3       0.01      0.05      0.01        22

    accuracy                           0.34      2352
   macro avg       0.30      0.30      0.24      2352
weighted avg       0.64      0.34      0.39      2352



## Balancing at the Algorithm Level

### Class Weight

In [47]:
param_list = {
    'max_depth': [None] + list(np.arange(2, 20)),
    'min_samples_split': [2, 5, 10, 20, 30, 50],
    'min_samples_leaf': [1, 5, 10, 20, 30, 50],
    'class_weight': [{0:1, 1:2, 2:3, 3:4}, {0:1, 1:5, 2:10, 3:20}, {0:1, 1:10, 2:25, 3:50}],
    'criterion': ['gini', 'entropy']
}

doDT(X_train,y_train)

{'min_samples_split': 30, 'min_samples_leaf': 10, 'max_depth': 3, 'criterion': 'gini', 'class_weight': {0: 1, 1: 5, 2: 10, 3: 20}} 0.34278305918545526
              precision    recall  f1-score   support

           0       0.82      0.48      0.61      1643
           1       0.21      0.27      0.24       433
           2       0.16      0.54      0.25       254
           3       0.00      0.00      0.00        22

    accuracy                           0.44      2352
   macro avg       0.30      0.32      0.27      2352
weighted avg       0.63      0.44      0.49      2352

